Rename sesuai folder kelas

In [1]:
import os

def rename_images(dataset_dir):

    for class_name in os.listdir(dataset_dir):

        class_path = os.path.join(dataset_dir, class_name)

        if os.path.isdir(class_path):

            image_files = sorted([
                f for f in os.listdir(class_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ])

            for idx, old_name in enumerate(image_files, start=1):

                ext = os.path.splitext(old_name)[1]

                new_name = f"{class_name}{idx}{ext}"

                old_path = os.path.join(class_path, old_name)
                new_path = os.path.join(class_path, new_name)

                os.rename(old_path, new_path)

                print(f"{old_name} -> {new_name}")


# Contoh penggunaan
dataset_dir = "/mnt/extended-home/dzakaaufa/dataset/baru_captioning"

rename_images(dataset_dir)

test_-10-_jpg.rf.540065d5f69c193736053f983f083f51.jpg -> tribusono1.jpg
test_-13-_jpg.rf.1e23ce806657d14c786849f6ee2c8855.jpg -> tribusono2.jpg
test_-17-_jpg.rf.2e046a3f51893b9d0514d63d628be360.jpg -> tribusono3.jpg
test_-30-_jpg.rf.3f2f3015da5331efd850a1d5033af931.jpg -> tribusono4.jpg
test_-33-_jpg.rf.032840420a7a3ff29baa3c1cfa0ba281.jpg -> tribusono5.jpg
test_-38-_jpg.rf.23ed7e067696d6989b761d12116bb2d3.jpg -> tribusono6.jpg
test_-39-_jpg.rf.2e433d205fdecf0d208daf1b5b1f3c87.jpg -> tribusono7.jpg
test_-47-_jpg.rf.2be04b6de4ff8f7293bc4d6c177aaf09.jpg -> tribusono8.jpg
test_-50-_jpg.rf.6f2c36f0442e0e04d33dde738f1436e4.jpg -> tribusono9.jpg
test_-58-_jpg.rf.c1df1a2d8e4b728969e68c6842b3d00b.jpg -> tribusono10.jpg
test_-6-_jpg.rf.2f34a7362051ea60c3490d29d4ae9e40.jpg -> tribusono11.jpg
test_-61-_jpg.rf.ac9352f4eb2fe4280ab5e79a8f6407de.jpg -> tribusono12.jpg
test_-63-_jpg.rf.32c6e7cd01ad134a342eb469291bc4c3.jpg -> tribusono13.jpg
test_-64-_jpg.rf.6fec5b719cb1124c3632b43593feda04.jpg -> trib

In [2]:
import os
import csv

def generate_csv(dataset_dir, output_csv):
    data = []

    # Loop setiap folder kelas
    for class_name in os.listdir(dataset_dir):
        class_path = os.path.join(dataset_dir, class_name)

        # Pastikan folder
        if os.path.isdir(class_path):

            # Ambil semua gambar
            image_files = sorted([
                f for f in os.listdir(class_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ])

            for file_name in image_files:

                # Path lengkap gambar
                file_path = os.path.join(class_path, file_name)

                data.append([
                    file_name,
                    file_path,
                    class_name
                ])

    # Simpan CSV
    with open(output_csv, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)

        # Header
        writer.writerow(['Nama', 'Image Path', 'CLASS'])

        # Isi data
        writer.writerows(data)

    print(f"CSV berhasil dibuat di: {output_csv}")


# Contoh penggunaan
dataset_dir = "/mnt/extended-home/dzakaaufa/dataset/baru_captioning"
output_csv = "/mnt/extended-home/dzakaaufa/dataset/caption/batik_dataset_baru.csv"

generate_csv(dataset_dir, output_csv)

CSV berhasil dibuat di: /mnt/extended-home/dzakaaufa/dataset/caption/batik_dataset_baru.csv


In [2]:
import pandas as pd
import re
from deep_translator import GoogleTranslator
import time

# Inisialisasi translator
translator = GoogleTranslator(source='id', target='en')

def safe_translate(text):
    """Fungsi pembungkus untuk mencegah script mati jika koneksi internet terputus."""
    try:
        time.sleep(0.1) # Jeda ringan agar tidak di-banned oleh Google
        return translator.translate(text).lower()
    except Exception as e:
        print(f"  [Warning] Gagal menerjemahkan '{text}': {e}")
        return text.lower() # Fallback ke bahasa asli jika gagal

def process_color_hybrid(color_str):
    """ Menangani campuran "Hijau muda" dan "Biru/ Blue/ 青い" """
    if pd.isna(color_str): return {'id': '', 'en': ''}
    
    color_str = str(color_str).replace('\n', '').strip()
    colors = color_str.split(',')
    
    id_list, en_list = [], []
    
    for c in colors:
        parts = [p.strip() for p in c.split('/')]
        id_color = parts[0].lower()
        id_list.append(id_color)
        
        # Jika ada terjemahan bawaan dari CSV (indeks 1)
        if len(parts) > 1 and parts[1]:
            en_list.append(parts[1].lower())
        else:
            # Jika tidak ada terjemahan di CSV, paksa translate pakai API
            en_list.append(safe_translate(id_color))
            
    return {
        'id': ' dan '.join(id_list) if len(id_list) > 1 else id_list[0],
        'en': ', '.join(en_list) if len(en_list) > 1 else en_list[0]
    }

def process_theme_hybrid(theme_str):
    """ Membersihkan noise kategori dan menerjemahkan tema """
    if pd.isna(theme_str): return {'id': '', 'en': ''}
    
    # Bersihkan noise: newline, kurung, dan label uppercase (CULTURE, FLORA, dll)
    clean_str = str(theme_str).replace('\n', '').replace('(', '').replace(')', '')
    clean_str = re.sub(r'[A-Z]{3,}', '', clean_str).strip()
    
    themes = clean_str.split('//')
    id_list, en_list = [], []
    
    for t in themes:
        parts = [p.strip() for p in t.split('/')]
        if not parts or not parts[0]: continue
        
        id_theme = parts[0].lower()
        id_list.append(id_theme)
        
        if len(parts) > 1 and parts[1]:
            en_list.append(parts[1].lower())
        else:
            en_list.append(safe_translate(id_theme))
            
    id_res = ', '.join(id_list[:-1]) + ' dan ' + id_list[-1] if len(id_list) > 1 else ''.join(id_list)
    en_res = ', '.join(en_list[:-1]) + ' and ' + en_list[-1] if len(en_list) > 1 else ''.join(en_list)
    
    return {'id': id_res, 'en': en_res}

def process_static_terms(text, category):
    """ Pembersihan istilah teknis yang lebih tangguh dengan regex """
    if pd.isna(text): return {'id': '', 'en': ''}
    text = str(text).lower().strip()
    result = {'id': text, 'en': text}
    
    if category == 'shape':
        if 'non' in text: result['en'] = 'non-geometric'
        elif 'geometris' in text: result['en'] = 'geometric'
        
    elif category == 'technique':
        if 'handprinting' in text: 
            result['id'] = 'cetak manual (handprinting)'
            result['en'] = 'handprinting'
        elif 'cap dan tulis' in text: result['en'] = 'stamped and hand-drawn'
        elif 'tulis' in text: 
            # Menangkap Tulis (Pindon) -> hand-drawn (Pindon)
            extra = re.search(r'\((.*?)\)', text)
            if extra: result['en'] = f"hand-drawn ({extra.group(1)})"
            else: result['en'] = 'hand-drawn'
        elif 'cap' in text: result['en'] = 'stamped'
            
    elif category == 'dye':
        if 'sintetis' in text or 'sintestis' in text:
            extra = re.search(r'\((.*?)\)', text)
            if extra: result['en'] = f"synthetic ({extra.group(1)})"
            else: result['en'] = 'synthetic'
        elif 'alam' in text: result['en'] = 'natural'
            
    return result

def generate_captions(input_csv, output_csv):
    df = pd.read_csv(input_csv)
    caption_id_list, caption_en_list = [], []
    
    print(f"Memproses {len(df)} baris data...")
    
    for index, row in df.iterrows():
        try:
            colors = process_color_hybrid(row['COLOR'])
            themes = process_theme_hybrid(row['THEME'])
            shape = process_static_terms(row['SHAPE'], 'shape')
            tech = process_static_terms(row['TECHNIQUE'], 'technique')
            dye = process_static_terms(row['DYE'], 'dye')
            
            batik_class = str(row['CLASS']).strip()
            fabric = str(row['FABRIC']).strip()
            history = str(row['HISTORY']).strip() if pd.notna(row['HISTORY']) else ""
            
            cap_en = (
                f"The batik fabric features the {batik_class} motif, "
                f"characterized by {colors['en']} motifs. "
                f"The dominant motifs include {themes['en']}. "
                f"The motifs are arranged in a {shape['en']} pattern across the fabric, "
                f"creating a harmonious and balanced composition."
            )
            
            # FORMAT INDONESIA (Fleksibel dengan Sejarah)
            cap_id = (f"{str(row['NAME']).strip()} merupakan mahakarya kelas Batik {batik_class} dengan tata letak {shape['id']}. "
                      f"Warna yang digunakan adalah {colors['id']}, membentuk motif utama {themes['id']}. "
                      f"Dibuat dengan teknik {tech['id']} menggunakan pewarna {dye['id']} pada kain {fabric}. "
                      f"Secara filosofis, {history}")
            
            caption_en_list.append(cap_en)
            caption_id_list.append(cap_id)
            print(f"  [OK] Baris {index} terproses.")
            
        except Exception as e:
            print(f"  [ERROR] Gagal pada baris {index}: {e}")
            caption_id_list.append("")
            caption_en_list.append("")
            
    df['CAPTION_ID'] = caption_id_list
    df['CAPTION_EN'] = caption_en_list
    
    base_dir = "/mnt/extended-home/dzakaaufa/dataset/image/"
    df['Image Path'] = df['CODE'].apply(lambda x: f"{base_dir}{str(x).strip()}.jpg")
    
    df.to_csv(output_csv, index=False)
    print(f"\nSelesai! File berhasil disimpan di {output_csv}")

generate_captions("/mnt/extended-home/dzakaaufa/dataset/caption/metadata_data_A.csv", "/mnt/extended-home/dzakaaufa/dataset/caption/data_A_inject.csv")

Memproses 327 baris data...
  [OK] Baris 0 terproses.
  [OK] Baris 1 terproses.
  [OK] Baris 2 terproses.
  [OK] Baris 3 terproses.
  [OK] Baris 4 terproses.
  [OK] Baris 5 terproses.
  [OK] Baris 6 terproses.
  [OK] Baris 7 terproses.
  [OK] Baris 8 terproses.
  [OK] Baris 9 terproses.
  [OK] Baris 10 terproses.
  [OK] Baris 11 terproses.
  [OK] Baris 12 terproses.
  [OK] Baris 13 terproses.
  [OK] Baris 14 terproses.
  [OK] Baris 15 terproses.
  [OK] Baris 16 terproses.
  [OK] Baris 17 terproses.
  [OK] Baris 18 terproses.
  [OK] Baris 19 terproses.
  [OK] Baris 20 terproses.
  [OK] Baris 21 terproses.
  [OK] Baris 22 terproses.
  [OK] Baris 23 terproses.
  [OK] Baris 24 terproses.
  [OK] Baris 25 terproses.
  [OK] Baris 26 terproses.
  [OK] Baris 27 terproses.
  [OK] Baris 28 terproses.
  [OK] Baris 29 terproses.
  [OK] Baris 30 terproses.
  [OK] Baris 31 terproses.
  [OK] Baris 32 terproses.
  [OK] Baris 33 terproses.
  [OK] Baris 34 terproses.
  [OK] Baris 35 terproses.
  [OK] Bar